In [0]:
dbutils.library.restartPython()

In [0]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print("src exists:", (project_root / "src").is_dir())

In [0]:
from pyspark.sql import functions as F
from src.config import config
from src.pipeline import process_batch

#### Build reproducible batches

Process deterministic batches A and B. The original project split on `2024-05-15`.

In [0]:
source_df = spark.table(config.source_transactions)
split_date = "2024-05-15"

batch_a_df = source_df.filter(
    (F.col("dateTime") < F.lit(split_date).cast("timestamp")) |
    F.col("dateTime").isNull()
)
batch_b_df = source_df.filter(
    F.col("dateTime") >= F.lit(split_date).cast("timestamp")
)

assert source_df.count() == batch_a_df.count() + batch_b_df.count()
print("Source:", source_df.count())
print("Batch A:", batch_a_df.count())
print("Batch B:", batch_b_df.count())

#### Run Batch A 

In [0]:
process_batch(spark=spark, batch_df=batch_a_df, batch_id="batch_a", config=config)

#### Run Batch B

In [0]:
process_batch(spark=spark, batch_df=batch_b_df, batch_id="batch_b", config=config)

#### Inspect state

In [0]:
spark.table(config.pipeline_audit).orderBy("_run_id").display()

print("Silver:", spark.table(config.silver_transactions).count())
print("Reject attempts:", spark.table(config.silver_invalid_transactions).count())
print("Gold:", spark.table(config.gold_daily_franchise_sales).count())